In [53]:
import requests
import pandas as pd
import xml.etree.ElementTree as ET
from pprint import pprint

## Fetching & Exploring an Bundestag plenary Seesion xml file
To start exploring we will try to fetch an xml file, and navigate it as a tree.
Once we are comfortable navigating the file we'll try flattening it into a flat table
/ a flat pandas df as the first step of denormilization (flattening a nested file structure),
so that we can after further normalize into different tables.

In [8]:
file_path ="https://www.bundestag.de/resource/blob/1140642/21057.xml"

response = requests.get(file_path)
response.raise_for_status()
print(response.status_code)
print(type(response))


200
<class 'requests.models.Response'>


In [27]:
# after fetching the xml using an https request
# we can access it as an tree by  converting it to
# an elementtree object, which is easier to navigate/work with
# for that we use pythons built in xml ElementTree object
# which represents an xml file, as the name suggests, as a tree
# of elements. We imported this type of object already so we can
# can make use of that class to instantiate an instance that holds
# the information of our plenary protocoll.

# For that we use the fromstring method, which expects, as the name suggests, as string
# as input. To access the our html response to a string, we can access its content attribute.

# Each tree has one core node, also referred to as the trunk of a tree.
# In this case we refer to it as the root of our element tree.
# We now pass our xml as a string, and get an elementree that we assign
# to root
# we see that we succesfully get an Elementree element (as the root is always returned, which holds
# all subelements/children of the rest of the tree), remember we're working with a hirarchical # datastructure here, we can access the root elements name by accessing its tag attribute,
# as well as see that the root holds 4 subelements which are its children
root = ET.fromstring(response.content)
print(type(root))
print(root.tag)
print(len(root))

<class 'xml.etree.ElementTree.Element'>
dbtplenarprotokoll
4


In [30]:
# in addition to that an element is of a certain type (its name), can hold on to values, usually text
# as well as can have attributes
# In our case our root element is called "dbtplenarprotokoll" which translates to "dbtplenaryprotocol"
# but holds varius attributes as metadata
# those attributes are returned as a python dictionary, which means that we can access
# them using their keys
# We will make use of that to get our plenary level metadata like session date, legislative
# period, session number etc
print(root.attrib)
print(type(root.attrib))
plenary_session_metadata = root.attrib

{'vertrieb': 'Bundesanzeiger Verlag GmbH, Postfach 1 0 05 34, 50445 Köln, Telefon (02 21) 97 66 83 40, Fax (02 21) 97 66 83 44, www.bundesanzeiger-verlag.de', 'herstellung': 'H. Heenemann GmbH  Co. KG, Buch- und Offsetdruckerei, Bessemerstraße 83–91, 12103 Berlin, www.heenemann-druck.de', 'sitzung-ort': 'Berlin', 'herausgeber': 'Deutscher Bundestag', 'issn': '0722-7980', 'wahlperiode': '21', 'sitzung-nr': '57', 'sitzung-datum': '30.01.2026', 'sitzung-start-uhrzeit': '09:00', 'sitzung-ende-uhrzeit': '15:00', 'sitzung-naechste-datum': '25.02.2026', 'start-seitennr': '6839'}
<class 'dict'>


In [32]:
# to test we try to access the legaslative period, session nr and
# date, later when flattening the file we can make use of this to
# add session level metadata to each speech
legaslative_period = plenary_session_metadata["wahlperiode"]
session_nr = plenary_session_metadata["sitzung-nr"]
session_date = plenary_session_metadata["sitzung-datum"]
print(f"Legaslative Period: {legaslative_period}")
print(f"Session Nr: {session_nr}")
print(f"Session Date: {session_date}")

Legaslative Period: 21
Session Nr: 57
Session Date: 30.01.2026


In [33]:
# ok now that we've covered everyhting of the root node
# we can move down the tree and explore its sub elements, also
# referred to as its child
for child in root:
    print(child.tag)

# in our case the root holds 4 children:
# the vorspann, the actual undergoings of the session in
# sitzungsverlauf which also hold the individual speeches,
# additional information in anlagen
# and lastly a speakerlist, which later might be useful to validate
# our extracted speeches against
# Of most interest to us is the sitzungsverlauf which contains
# the core of our speeches


vorspann
sitzungsverlauf
anlagen
rednerliste


In [36]:
# we can access the individual subelements
# using their index location, to make things more readable
# and explore subelements individually we assign them to new variables
opening = root[0]
main_session_part = root[1]
attachement = root[2]
speaker_list = root[3]
print(opening.tag)
print(main_session_part.tag)
print(attachement.tag)
print(speaker_list.tag)

vorspann
sitzungsverlauf
anlagen
rednerliste


In [46]:
# As the core of the session of our interest is the main part
# as it holds all our speeches, lets dig deeper and check which subelements
# it consists of
print(f"Nr. of main elements: {len(main_session_part)}\n")

print("Main elements (Name and attributes): \n")
for child in main_session_part:
     print(child.tag)
     print(child.attrib)

# What we see is that the main part is wrapped inside a session beginning
# and session ending element, with agenda items in between
# We see that the main session wrapper hold the start and end time as attributes
# The agenda items the agenda item names
# The start and end time, match those of the session level metadata we,
# explored earlier
# I've mannually explored the opening and closing elements in my browser,
# hich showed that those seem to hold opening remarks by the BT president
# so for now is of lesser interest
# Lets grab an agenda item to dig deeper and explore its structure

Nr. of main elements: 9

Main elements (Name and attributes): 

sitzungsbeginn
{'sitzung-start-uhrzeit': '09:00'}
tagesordnungspunkt
{'top-id': 'Tagesordnungspunkt 7'}
tagesordnungspunkt
{'top-id': 'Tagesordnungspunkt 24'}
tagesordnungspunkt
{'top-id': 'Tagesordnungspunkt'}
tagesordnungspunkt
{'top-id': 'Zusatzpunkt 8'}
tagesordnungspunkt
{'top-id': 'Tagesordnungspunkt 5'}
tagesordnungspunkt
{'top-id': 'Tagesordnungspunkt 26'}
tagesordnungspunkt
{'top-id': 'Zusatzpunkt 9'}
sitzungsende
{'sitzung-ende-uhrzeit': '15:00'}


In [49]:
agenda_item_one = main_session_part[1]
print(f"Agenda item name: {agenda_item_one.tag}")
print(f"Agenda attributes: {agenda_item_one.attrib}")
print(f"Nr. of sub elements: {len(agenda_item_one)}")

Agenda item name: tagesordnungspunkt
Agenda attributes: {'top-id': 'Tagesordnungspunkt 7'}
Nr. of sub elements: 33


In [67]:
# lets check what sub elements
# the agenda item holds
for item in agenda_item_one:
    print(item.tag)


# we see that the item mainly contains
# paragraph elements (marked by "p"),
# a few comments ("kommentar") as well
# as speeches

# After manually inspecting the agenda item in the browser it becomes
# clear that the agenda item, similar to what we saw before in the agenda items,
# is wrapped inside an opening and closing part. This time the opening contains
# an opening by the president, an agenda item, possibly agenda sub items and eventually documents
# that the debate relates to.

# Lets try to isolate those paragraph elements to extraxt those items
# Before moving on to individual speeches


p
p
p
p
p
p
p
p
p
kommentar
rede
rede
rede
rede
rede
rede
rede
rede
rede
rede
rede
rede
rede
name
p
p
kommentar
rede
rede
name
p
p
p


In [72]:
# At this point we make use of a method that
# returns all elements that follow a certain
# tag name, that method is called findall. If you parsed html
# before you may have stumbled upon this method before.
# In our case we make use of findall, to get all paragraph
# elements part of the first agenda item
agenda_paragraphs = agenda_item_one.findall("p")
for par in agenda_paragraphs:
    print(par)
    print(par.attrib)

<Element 'p' at 0x11798d670>
{'klasse': 'J'}
<Element 'p' at 0x11798d6c0>
{'klasse': 'T_NaS'}
<Element 'p' at 0x11798d710>
{'klasse': 'T_fett'}
<Element 'p' at 0x11798d7b0>
{'klasse': 'T_NaS'}
<Element 'p' at 0x11798d800>
{'klasse': 'T_fett'}
<Element 'p' at 0x11798d850>
{'klasse': 'T_Drs'}
<Element 'p' at 0x11798d8f0>
{'klasse': 'T_Ueberweisung'}
<Element 'p' at 0x11798d940>
{'klasse': 'J'}
<Element 'p' at 0x11798d990>
{'klasse': 'J'}
<Element 'p' at 0x11795df80>
{'klasse': 'J_1'}
<Element 'p' at 0x11795dfd0>
{'klasse': 'J'}
<Element 'p' at 0x1179607c0>
{'klasse': 'J_1'}
<Element 'p' at 0x117960810>
{'klasse': 'J'}
<Element 'p' at 0x117960860>
{'klasse': 'J'}


In [65]:


speeches_first_agenda_item = agenda_item_one.findall("rede")
for speech in speeches_first_agenda_item:
    print(speech.tag,speech.attrib)

rede {'id': 'ID215700100'}
rede {'id': 'ID215700200'}
rede {'id': 'ID215700300'}
rede {'id': 'ID215700400'}
rede {'id': 'ID215700500'}
rede {'id': 'ID215700600'}
rede {'id': 'ID215700700'}
rede {'id': 'ID215700800'}
rede {'id': 'ID215700900'}
rede {'id': 'ID215701000'}
rede {'id': 'ID215701100'}
rede {'id': 'ID215701200'}
rede {'id': 'ID215701300'}
rede {'id': 'ID215701400'}
rede {'id': 'ID215701500'}


In [63]:
for x in agenda_item_one.iter("rede"):
    print(x.find("vorname"))

None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
